In [1]:
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# Connexion au Data Warehouse
conn = sqlite3.connect('datawarehouse.db')

# Affichage des tables et vues disponibles
tables = pd.read_sql("""
    SELECT type, name 
    FROM sqlite_master 
    WHERE type IN ('table','view') 
    ORDER BY type, name
""", conn)
print('=== Structure du Data Warehouse ===')
print(tables.to_string(index=False))

=== Structure du Data Warehouse ===
 type                        name
table                  Dim_Client
table                   Dim_Temps
table        Dim_Type_Transaction
table            Fait_Transaction
 view  Anomalies_Par_Destinataire
 view      Anomalies_Par_Emetteur
 view      Cube_Fraude_Type_Temps
 view Evolution_Fraude_Temporelle


---
## 0. Indicateurs globaux (vue d'ensemble)

In [2]:
# ─── INDICATEURS GLOBAUX ───────────────────────────────────────────────────
# Vue synthétique sur l'ensemble des transactions du cube

df_global = pd.read_sql("""
    SELECT
        COUNT(*)                                              AS total_transactions,
        SUM(isFraud)                                         AS total_fraudes,
        ROUND(100.0 * SUM(isFraud) / COUNT(*), 4)            AS taux_fraude_pct,
        ROUND(SUM(amount), 2)                                AS montant_total,
        ROUND(AVG(amount), 2)                                AS montant_moyen,
        SUM(flag_incoherence_solde)                          AS nb_incoherences_solde,
        SUM(solde_vide_apres)                                AS nb_soldes_vides,
        SUM(flag_montant_aberrant)                           AS nb_montants_aberrants
    FROM Fait_Transaction
""", conn)

print('=== Indicateurs globaux du cube ===')
print(df_global.T.to_string(header=False))

=== Indicateurs globaux du cube ===
total_transactions     6.985700e+04
total_fraudes          1.070000e+02
taux_fraude_pct        1.532000e-01
montant_total          1.145794e+10
montant_moyen          1.640200e+05
nb_incoherences_solde  4.946200e+04
nb_soldes_vides        3.577300e+04
nb_montants_aberrants  2.561000e+03


---
## 1. SLICE — Analyse d'une tranche de dimension

**Concept MDX :** On fixe une valeur sur une dimension et on projette les mesures sur les dimensions restantes.  
**Équivalent MDX :**
```
SELECT {[Measures].[Nb_Fraudes], [Measures].[Taux_Fraude]} ON COLUMNS,
       {[Type_Transaction].[Members]} ON ROWS
FROM [Cube_Fraude]
WHERE ([Tranche_Horaire].[nuit])
```

In [3]:
# ─── SLICE 1 : Transactions nocturnes uniquement ───────────────────────────
# On fixe la dimension Temps sur la valeur 'nuit'
# et on analyse les types de transactions

df_slice_nuit = pd.read_sql("""
    SELECT
        type_transaction,
        nb_transactions,
        nb_fraudes,
        taux_fraude_pct,
        ROUND(montant_total, 2)  AS montant_total,
        ROUND(montant_moyen, 2)  AS montant_moyen
    FROM Cube_Fraude_Type_Temps
    WHERE tranche_horaire = 'nuit'
      AND type_transaction NOT IN ('Tous')
    ORDER BY nb_fraudes DESC
""", conn)

print('=== SLICE — Transactions nocturnes par type ===')
print(df_slice_nuit.to_string(index=False))
print(f"\nObservation : Le TRANSFER nocturne affiche un taux de fraude de "
      f"{df_slice_nuit[df_slice_nuit.type_transaction=='TRANSFER'].taux_fraude_pct.values[0]}%, "
      f"soit le plus élevé de la tranche nuit.")

=== SLICE — Transactions nocturnes par type ===
type_transaction  nb_transactions  nb_fraudes  taux_fraude_pct  montant_total  montant_moyen
        CASH_OUT              682          23             3.37   120430980.11      176585.01
        TRANSFER              464          21             4.53   204631676.07      441016.54
         CASH_IN             1121           0             0.00   187786674.48      167517.10
           DEBIT              284           0             0.00     1059738.49        3731.47
         PAYMENT             2953           0             0.00    19193718.76        6499.74

Observation : Le TRANSFER nocturne affiche un taux de fraude de 4.53%, soit le plus élevé de la tranche nuit.


In [4]:
# ─── SLICE 2 : Types à risque uniquement (TRANSFER + CASH_OUT) ─────────────
# On fixe la dimension Type sur les seuls types générant de la fraude
# et on analyse toutes les tranches horaires

df_slice_type = pd.read_sql("""
    SELECT
        tranche_horaire,
        type_transaction,
        nb_transactions,
        nb_fraudes,
        taux_fraude_pct,
        ROUND(montant_total, 2) AS montant_total
    FROM Cube_Fraude_Type_Temps
    WHERE type_transaction IN ('TRANSFER', 'CASH_OUT')
      AND tranche_horaire NOT IN ('Toutes')
    ORDER BY taux_fraude_pct DESC
""", conn)

print('=== SLICE — Types TRANSFER et CASH_OUT par tranche horaire ===')
print(df_slice_type.to_string(index=False))

=== SLICE — Types TRANSFER et CASH_OUT par tranche horaire ===
tranche_horaire type_transaction  nb_transactions  nb_fraudes  taux_fraude_pct  montant_total
           nuit         TRANSFER              464          21             4.53   2.046317e+08
           nuit         CASH_OUT              682          23             3.37   1.204310e+08
          matin         TRANSFER             5709          32             0.56   4.817931e+09
          matin         CASH_OUT            19707          31             0.16   3.681748e+09


---
## 2. DICE — Filtrage multi-dimensionnel

**Concept MDX :** On applique des filtres sur plusieurs dimensions simultanément pour isoler un sous-cube précis.  
**Équivalent MDX :**
```
SELECT {[Measures].[Nb_Fraudes], [Measures].[Taux_Fraude]} ON COLUMNS,
       CrossJoin({[Type].[TRANSFER],[Type].[CASH_OUT]},
                 {[Tranche_Horaire].[nuit]}) ON ROWS
FROM [Cube_Fraude]
```

In [5]:
# ─── DICE 1 : TRANSFER + CASH_OUT nocturnes (sous-cube haute fraude) ────────
# Intersection de deux filtres : type de transaction ET tranche horaire

df_dice1 = pd.read_sql("""
    SELECT
        type_transaction,
        tranche_horaire,
        nb_transactions,
        nb_fraudes,
        taux_fraude_pct,
        ROUND(montant_total, 2)  AS montant_total,
        ROUND(montant_moyen, 2)  AS montant_moyen
    FROM Cube_Fraude_Type_Temps
    WHERE type_transaction IN ('TRANSFER', 'CASH_OUT')
      AND tranche_horaire = 'nuit'
    ORDER BY taux_fraude_pct DESC
""", conn)

print('=== DICE 1 — TRANSFER + CASH_OUT, tranche nuit ===')
print(df_dice1.to_string(index=False))

=== DICE 1 — TRANSFER + CASH_OUT, tranche nuit ===
type_transaction tranche_horaire  nb_transactions  nb_fraudes  taux_fraude_pct  montant_total  montant_moyen
        TRANSFER            nuit              464          21             4.53   204631676.07      441016.54
        CASH_OUT            nuit              682          23             3.37   120430980.11      176585.01


In [6]:
# ─── DICE 2 : Émetteurs suspects avec anomalies multiples ──────────────────
# Filtrage sur plusieurs indicateurs d'anomalie simultanément
# Émetteurs ayant au moins 1 fraude ET solde vidé ET incohérence de solde

df_dice2 = pd.read_sql("""
    SELECT
        nom_emetteur,
        nb_fraudes,
        nb_incoherence_solde,
        nb_solde_vide,
        nb_ratio_depassement,
        ROUND(montant_total_emis, 2) AS montant_total_emis
    FROM Anomalies_Par_Emetteur
    WHERE nb_fraudes >= 1
      AND nb_solde_vide >= 1
      AND nb_incoherence_solde >= 1
    ORDER BY nb_fraudes DESC, montant_total_emis DESC
    LIMIT 15
""", conn)

print('=== DICE 2 — Émetteurs avec fraude + solde vidé + incohérence ===')
print(df_dice2.to_string(index=False))
print(f"\n→ {len(df_dice2)} émetteurs présentent ces 3 anomalies simultanément.")

=== DICE 2 — Émetteurs avec fraude + solde vidé + incohérence ===
nom_emetteur  nb_fraudes  nb_incoherence_solde  nb_solde_vide  nb_ratio_depassement  montant_total_emis
 C1026280121           1                     1              1                     0          1078013.76
  C749981943           1                     1              1                     0           416001.33
 C2102265902           1                     1              1                     0           181728.11
   C13692003           1                     1              1                     1           132842.64

→ 4 émetteurs présentent ces 3 anomalies simultanément.


---
## 3. DRILL-DOWN — Du général vers le détail

**Concept MDX :** Naviguer dans la hiérarchie d'une dimension, du niveau agrégé vers le niveau le plus fin.  
**Équivalent MDX :**
```
DRILLDOWNLEVEL(
  {[Type_Transaction].[Tous]},
  [Type_Transaction]
)
```

In [7]:
# ─── DRILL-DOWN : Total → Par type → Par type + tranche horaire ─────────────
# Niveau 1 : vue globale (tous types, toutes tranches)

df_n1 = pd.read_sql("""
    SELECT 'TOTAL' AS niveau, 'Tous types' AS type_transaction,
           'Toutes tranches' AS tranche_horaire,
           nb_transactions, nb_fraudes, taux_fraude_pct, montant_total
    FROM Cube_Fraude_Type_Temps
    WHERE type_transaction = 'Tous' AND tranche_horaire = 'Toutes'
""", conn)

# Niveau 2 : détail par type
df_n2 = pd.read_sql("""
    SELECT 'TYPE' AS niveau, type_transaction,
           'Toutes tranches' AS tranche_horaire,
           nb_transactions, nb_fraudes, taux_fraude_pct, montant_total
    FROM Cube_Fraude_Type_Temps
    WHERE tranche_horaire = 'Toutes'
      AND type_transaction != 'Tous'
    ORDER BY nb_fraudes DESC
""", conn)

# Niveau 3 : détail par type ET tranche horaire
df_n3 = pd.read_sql("""
    SELECT 'TYPE+HEURE' AS niveau, type_transaction, tranche_horaire,
           nb_transactions, nb_fraudes, taux_fraude_pct, montant_total
    FROM Cube_Fraude_Type_Temps
    WHERE type_transaction NOT IN ('Tous')
      AND tranche_horaire NOT IN ('Toutes')
    ORDER BY nb_fraudes DESC
""", conn)

print('=== DRILL-DOWN : Niveau 1 — Total général ===')
print(df_n1[['niveau','type_transaction','nb_transactions','nb_fraudes','taux_fraude_pct']].to_string(index=False))

print('\n=== DRILL-DOWN : Niveau 2 — Par type de transaction ===')
print(df_n2[['niveau','type_transaction','nb_transactions','nb_fraudes','taux_fraude_pct']].to_string(index=False))

print('\n=== DRILL-DOWN : Niveau 3 — Par type ET tranche horaire ===')
print(df_n3[['niveau','type_transaction','tranche_horaire','nb_transactions','nb_fraudes','taux_fraude_pct']].to_string(index=False))

=== DRILL-DOWN : Niveau 1 — Total général ===
niveau type_transaction  nb_transactions  nb_fraudes  taux_fraude_pct
 TOTAL       Tous types            69857         107             0.15

=== DRILL-DOWN : Niveau 2 — Par type de transaction ===
niveau type_transaction  nb_transactions  nb_fraudes  taux_fraude_pct
  TYPE         CASH_OUT            20389          54             0.26
  TYPE         TRANSFER             6173          53             0.86
  TYPE          CASH_IN            13785           0             0.00
  TYPE            DEBIT              778           0             0.00
  TYPE          PAYMENT            28732           0             0.00

=== DRILL-DOWN : Niveau 3 — Par type ET tranche horaire ===
    niveau type_transaction tranche_horaire  nb_transactions  nb_fraudes  taux_fraude_pct
TYPE+HEURE         TRANSFER           matin             5709          32             0.56
TYPE+HEURE         CASH_OUT           matin            19707          31             0.16
TYPE+H

---
## 4. ROLL-UP — Agrégation montante

**Concept MDX :** Opération inverse du drill-down. On consolide les données en remontant dans la hiérarchie.  
**Équivalent MDX :**
```
SELECT {[Measures].[Nb_Fraudes]} ON COLUMNS,
       {[Tranche_Horaire].[Toutes]} ON ROWS
FROM [Cube_Fraude]
```

In [8]:
# ─── ROLL-UP 1 : Agrégation par tranche horaire (toutes tranches) ───────────
# On remonte de (type × tranche) vers (tranche uniquement)

df_rollup1 = pd.read_sql("""
    SELECT
        tranche_horaire,
        SUM(nb_transactions)                                   AS total_transactions,
        SUM(nb_fraudes)                                        AS total_fraudes,
        ROUND(100.0 * SUM(nb_fraudes) / SUM(nb_transactions), 4) AS taux_fraude_pct,
        ROUND(SUM(montant_total), 2)                           AS montant_total
    FROM Cube_Fraude_Type_Temps
    WHERE type_transaction NOT IN ('Tous')
      AND tranche_horaire NOT IN ('Toutes')
    GROUP BY tranche_horaire
    ORDER BY total_fraudes DESC
""", conn)

print('=== ROLL-UP 1 — Agrégation par tranche horaire ===')
print(df_rollup1.to_string(index=False))

=== ROLL-UP 1 — Agrégation par tranche horaire ===
tranche_horaire  total_transactions  total_fraudes  taux_fraude_pct  montant_total
          matin               64353             63           0.0979   1.092484e+10
           nuit                5504             44           0.7994   5.331028e+08


In [9]:
# ─── ROLL-UP 2 : Agrégation par catégorie de type ───────────────────────────
# On remonte de type précis vers catégorie (sortant / entrant / interne)

df_rollup2 = pd.read_sql("""
    SELECT
        tt.categorie,
        COUNT(f.id_transaction)                                    AS nb_transactions,
        SUM(f.isFraud)                                             AS nb_fraudes,
        ROUND(100.0 * SUM(f.isFraud) / COUNT(f.id_transaction), 4) AS taux_fraude_pct,
        ROUND(SUM(f.amount), 2)                                    AS montant_total
    FROM Fait_Transaction f
    JOIN Dim_Type_Transaction tt ON f.id_type = tt.id_type
    GROUP BY tt.categorie
    ORDER BY nb_fraudes DESC
""", conn)

print('=== ROLL-UP 2 — Par catégorie de transaction (entrant/sortant/interne) ===')
print(df_rollup2.to_string(index=False))
print("\nObservation : Les transactions 'interne' (TRANSFER + CASH_OUT) "
      "concentrent 100% des fraudes détectées.")

=== ROLL-UP 2 — Par catégorie de transaction (entrant/sortant/interne) ===
categorie  nb_transactions  nb_fraudes  taux_fraude_pct  montant_total
  interne            26562         107           0.4028   8.824741e+09
  sortant            29510           0           0.0000   3.044194e+08
  entrant            13785           0           0.0000   2.328782e+09

Observation : Les transactions 'interne' (TRANSFER + CASH_OUT) concentrent 100% des fraudes détectées.


---
## 5. PIVOT / Cross-tab — Vue croisée multidimensionnelle

**Concept MDX :** Présentation matricielle croisant deux dimensions sur les axes des colonnes et des lignes.  
**Équivalent MDX :**
```
SELECT {[Tranche_Horaire].[matin],[Tranche_Horaire].[nuit]} ON COLUMNS,
       {[Type_Transaction].[Members]} ON ROWS
FROM [Cube_Fraude]
WHERE [Measures].[Taux_Fraude_Pct]
```

In [10]:
# ─── PIVOT : Taux de fraude (type × tranche horaire) ────────────────────────

df_pivot_raw = pd.read_sql("""
    SELECT type_transaction, tranche_horaire, taux_fraude_pct
    FROM Cube_Fraude_Type_Temps
    WHERE type_transaction NOT IN ('Tous')
      AND tranche_horaire NOT IN ('Toutes')
""", conn)

df_pivot = df_pivot_raw.pivot(
    index='type_transaction',
    columns='tranche_horaire',
    values='taux_fraude_pct'
).fillna(0)

# Ajouter colonne total (toutes tranches)
df_tot = pd.read_sql("""
    SELECT type_transaction, taux_fraude_pct AS total_toutes_tranches
    FROM Cube_Fraude_Type_Temps
    WHERE tranche_horaire = 'Toutes' AND type_transaction != 'Tous'
""", conn).set_index('type_transaction')

df_pivot = df_pivot.join(df_tot)

print('=== PIVOT — Taux de fraude (%) par type × tranche horaire ===')
print(df_pivot.to_string())
print("\nLecture : Les cellules non nulles signalent les zones à risque.")

=== PIVOT — Taux de fraude (%) par type × tranche horaire ===
                  matin  nuit  total_toutes_tranches
type_transaction                                    
CASH_IN            0.00  0.00                   0.00
CASH_OUT           0.16  3.37                   0.26
DEBIT              0.00  0.00                   0.00
PAYMENT            0.00  0.00                   0.00
TRANSFER           0.56  4.53                   0.86

Lecture : Les cellules non nulles signalent les zones à risque.


In [11]:
# ─── PIVOT : Nombre de fraudes (type × tranche horaire) ─────────────────────

df_pivot_nb = pd.read_sql("""
    SELECT type_transaction, tranche_horaire, nb_fraudes
    FROM Cube_Fraude_Type_Temps
    WHERE type_transaction NOT IN ('Tous')
      AND tranche_horaire NOT IN ('Toutes')
""", conn).pivot(
    index='type_transaction',
    columns='tranche_horaire',
    values='nb_fraudes'
).fillna(0).astype(int)

df_pivot_nb['TOTAL'] = df_pivot_nb.sum(axis=1)

print('=== PIVOT — Nombre de fraudes par type × tranche horaire ===')
print(df_pivot_nb.to_string())

=== PIVOT — Nombre de fraudes par type × tranche horaire ===
tranche_horaire   matin  nuit  TOTAL
type_transaction                    
CASH_IN               0     0      0
CASH_OUT             31    23     54
DEBIT                 0     0      0
PAYMENT               0     0      0
TRANSFER             32    21     53


---
## 6. Mesure calculée — Score de risque composite

**Concept MDX :** Création d'une nouvelle mesure dérivée combinant plusieurs indicateurs existants.  
**Équivalent MDX :**
```
MEMBER [Measures].[Score_Risque] AS
    [Measures].[Nb_Fraudes] * 10
  + [Measures].[Nb_Ratio_Depassement] * 3
  + [Measures].[Nb_Incoherence_Solde] * 2
  + [Measures].[Nb_Solde_Vide] * 1
```

In [12]:
# ─── MESURE CALCULÉE 1 : Score de risque par émetteur ────────────────────────
# Pondération : fraude confirmée (×10) > ratio dépassement (×3) 
#               > incohérence solde (×2) > solde vidé (×1)

df_score = pd.read_sql("""
    SELECT
        nom_emetteur,
        nb_fraudes,
        nb_ratio_depassement,
        nb_incoherence_solde,
        nb_solde_vide,
        ROUND(montant_total_emis, 2)  AS montant_total_emis,
        -- Mesure calculée : score de risque composite
        (nb_fraudes * 10
         + nb_ratio_depassement * 3
         + nb_incoherence_solde * 2
         + nb_solde_vide * 1
        ) AS score_risque
    FROM Anomalies_Par_Emetteur
    WHERE nb_fraudes > 0 OR nb_ratio_depassement > 0
    ORDER BY score_risque DESC
    LIMIT 20
""", conn)

print('=== MESURE CALCULÉE — Top 20 émetteurs par score de risque composite ===')
print(df_score.to_string(index=False))

=== MESURE CALCULÉE — Top 20 émetteurs par score de risque composite ===
nom_emetteur  nb_fraudes  nb_ratio_depassement  nb_incoherence_solde  nb_solde_vide  montant_total_emis  score_risque
   C13692003           1                     1                     1              1           132842.64            16
  C749981943           1                     0                     1              1           416001.33            13
 C2102265902           1                     0                     1              1           181728.11            13
 C1026280121           1                     0                     1              1          1078013.76            13
 C1305486145           1                     0                     0              1              181.00            11
  C840083671           1                     0                     0              1              181.00            11
 C1420196421           1                     0                     0              1             2806.

In [13]:
# ─── MESURE CALCULÉE 2 : Score de suspicion destinataire ────────────────────
# Un destinataire est suspect s'il reçoit des transactions frauduleuses
# et/ou des montants aberrants

df_score_dest = pd.read_sql("""
    SELECT
        nom_destinataire,
        nb_transactions_recues,
        nb_fraudes_associees,
        nb_montants_aberrants,
        ROUND(montant_total_recu, 2) AS montant_total_recu,
        -- Score : fraude pèse plus lourd que montant aberrant
        (nb_fraudes_associees * 10 + nb_montants_aberrants * 4) AS score_suspicion
    FROM Anomalies_Par_Destinataire
    WHERE nb_fraudes_associees > 0
    ORDER BY score_suspicion DESC
    LIMIT 15
""", conn)

print('=== MESURE CALCULÉE — Top 15 destinataires par score de suspicion ===')
print(df_score_dest.to_string(index=False))

=== MESURE CALCULÉE — Top 15 destinataires par score de suspicion ===
nom_destinataire  nb_transactions_recues  nb_fraudes_associees  nb_montants_aberrants  montant_total_recu  score_suspicion
      C803116137                      43                     1                     11         23488297.91               54
      C716083600                      49                     1                      7         23670606.41               38
      C932583850                      47                     1                      6         17159677.20               34
      C991505714                      31                     1                      6         17878031.24               34
      C766681183                      39                     1                      6         24060320.76               34
     C1570256460                      20                     1                      6         20888528.81               34
     C1394526584                      25                     1       

---
## 7. TOP-N — Classement des entités les plus suspectes

**Concept MDX :** Sélection des N premiers membres d'une dimension selon une mesure donnée.  
**Équivalent MDX :**
```
TOPCOUNT(
  [Client].[Emetteur].[Members],
  10,
  [Measures].[Montant_Total_Emis]
)
```

In [14]:
# ─── TOP-N 1 : Top 10 émetteurs par montant total frauduleux ────────────────

df_top_emetteur = pd.read_sql("""
    SELECT
        nom_emetteur,
        nb_fraudes,
        ROUND(montant_total_emis, 2)  AS montant_frauduleux,
        ROUND(montant_moyen, 2)       AS montant_moyen_transaction
    FROM Anomalies_Par_Emetteur
    WHERE nb_fraudes >= 1
    ORDER BY montant_total_emis DESC
    LIMIT 10
""", conn)

print('=== TOP-N — Top 10 émetteurs frauduleux par montant ===')
print(df_top_emetteur.to_string(index=False))

=== TOP-N — Top 10 émetteurs frauduleux par montant ===
nom_emetteur  nb_fraudes  montant_frauduleux  montant_moyen_transaction
    C7162498           1         10000000.00                10000000.00
  C351297720           1         10000000.00                10000000.00
  C666654362           1          5460002.91                 5460002.91
 C1588880909           1          5460002.91                 5460002.91
 C2047521920           1          2930418.44                 2930418.44
 C2044690596           1          2930418.44                 2930418.44
  C394488466           1          2539898.07                 2539898.07
  C728718059           1          2539898.07                 2539898.07
 C1334405552           1          1277212.77                 1277212.77
  C467632528           1          1277212.77                 1277212.77


In [15]:
# ─── TOP-N 2 : Top 10 destinataires par nombre de montants aberrants ─────────

df_top_dest = pd.read_sql("""
    SELECT
        nom_destinataire,
        nb_transactions_recues,
        nb_fraudes_associees,
        nb_montants_aberrants,
        ROUND(montant_total_recu, 2)  AS montant_total_recu,
        ROUND(montant_moyen_recu, 2)  AS montant_moyen_recu
    FROM Anomalies_Par_Destinataire
    ORDER BY nb_montants_aberrants DESC, nb_fraudes_associees DESC
    LIMIT 10
""", conn)

print('=== TOP-N — Top 10 destinataires par montants aberrants reçus ===')
print(df_top_dest.to_string(index=False))

=== TOP-N — Top 10 destinataires par montants aberrants reçus ===
nom_destinataire  nb_transactions_recues  nb_fraudes_associees  nb_montants_aberrants  montant_total_recu  montant_moyen_recu
     C1590550415                      67                     0                     13         34220739.67           510757.31
      C665576141                      62                     0                     12         30907063.03           498501.02
      C803116137                      43                     1                     11         23488297.91           546239.49
      C248609774                      63                     0                     10         28305113.44           449287.51
       C97730845                      54                     0                     10         26258916.76           486276.24
     C1286084959                      69                     0                      9         24917402.71           361121.78
      C451111351                      56            

---
## 8. Analyse temporelle — Évolution des fraudes

Étude de l'évolution du taux de fraude au fil des steps de simulation.

In [16]:
# ─── ÉVOLUTION TEMPORELLE ───────────────────────────────────────────────────

df_evol = pd.read_sql("""
    SELECT
        step,
        tranche_horaire,
        nb_transactions,
        nb_fraudes,
        taux_fraude_pct,
        ROUND(montant_total, 2) AS montant_total
    FROM Evolution_Fraude_Temporelle
    ORDER BY step
""", conn)

print('=== ÉVOLUTION TEMPORELLE — Fraudes par step ===')
print(df_evol.to_string(index=False))

step_max = df_evol.loc[df_evol.taux_fraude_pct.idxmax()]
print(f"\nPic de fraude : Step {int(step_max.step)} "
      f"({step_max.tranche_horaire}) → taux = {step_max.taux_fraude_pct}%")

=== ÉVOLUTION TEMPORELLE — Fraudes par step ===
 step tranche_horaire  nb_transactions  nb_fraudes  taux_fraude_pct  montant_total
    1            nuit             2708          16             0.59   2.854292e+08
    2            nuit             1014           8             0.79   8.592160e+07
    3            nuit              552           4             0.72   4.329388e+07
    4            nuit              565          10             1.77   7.291003e+07
    5            nuit              665           6             0.90   4.554809e+07
    6           matin             1660          22             1.33   1.643106e+08
    7           matin             6837          12             0.18   8.329308e+08
    8           matin            21097          12             0.06   3.439602e+09
    9           matin            34759          17             0.05   6.487996e+09

Pic de fraude : Step 4 (nuit) → taux = 1.77%


In [17]:
# ─── ANALYSE : Fraudes nocturnes vs matinales ────────────────────────────────

df_nuit_vs_matin = pd.read_sql("""
    SELECT
        tranche_horaire,
        SUM(nb_transactions)  AS total_transactions,
        SUM(nb_fraudes)       AS total_fraudes,
        ROUND(100.0 * SUM(nb_fraudes) / SUM(nb_transactions), 4) AS taux_moyen_fraude,
        ROUND(AVG(taux_fraude_pct), 4)  AS taux_fraude_moyen_par_step
    FROM Evolution_Fraude_Temporelle
    GROUP BY tranche_horaire
""", conn)

print('=== Comparaison nuit vs matin ===')
print(df_nuit_vs_matin.to_string(index=False))

=== Comparaison nuit vs matin ===
tranche_horaire  total_transactions  total_fraudes  taux_moyen_fraude  taux_fraude_moyen_par_step
          matin               64353             63             0.0979                       0.405
           nuit                5504             44             0.7994                       0.954


---
## 9. Export des résultats

In [18]:
# ─── EXPORT CSV ─────────────────────────────────────────────────────────────

exports = {
    'slice_nuit.csv':              df_slice_nuit,
    'slice_types_a_risque.csv':    df_slice_type,
    'dice_transfer_cashout.csv':   df_dice1,
    'dice_emetteurs_multiples.csv':df_dice2,
    'rollup_tranche_horaire.csv':  df_rollup1,
    'rollup_categorie.csv':        df_rollup2,
    'pivot_taux_fraude.csv':       df_pivot,
    'score_risque_emetteurs.csv':  df_score,
    'score_suspicion_dest.csv':    df_score_dest,
    'top10_emetteurs.csv':         df_top_emetteur,
    'top10_destinataires.csv':     df_top_dest,
    'evolution_temporelle.csv':    df_evol,
}

for nom, df in exports.items():
    df.to_csv(nom, index=False)
    print(f'✔ Exporté : {nom} ({len(df)} lignes)')

conn.close()
print('\n✔ Connexion fermée. Tous les fichiers exportés.')

✔ Exporté : slice_nuit.csv (5 lignes)
✔ Exporté : slice_types_a_risque.csv (4 lignes)
✔ Exporté : dice_transfer_cashout.csv (2 lignes)
✔ Exporté : dice_emetteurs_multiples.csv (4 lignes)
✔ Exporté : rollup_tranche_horaire.csv (2 lignes)
✔ Exporté : rollup_categorie.csv (3 lignes)
✔ Exporté : pivot_taux_fraude.csv (5 lignes)
✔ Exporté : score_risque_emetteurs.csv (20 lignes)
✔ Exporté : score_suspicion_dest.csv (15 lignes)
✔ Exporté : top10_emetteurs.csv (10 lignes)
✔ Exporté : top10_destinataires.csv (10 lignes)
✔ Exporté : evolution_temporelle.csv (9 lignes)

✔ Connexion fermée. Tous les fichiers exportés.


---
## Synthèse des résultats

| Opération | Résultat clé |
|-----------|-------------|
| **Slice nuit** | TRANSFER nocturne : taux de fraude 4,53% (vs 0,56% le matin) |
| **Slice type** | Seuls TRANSFER et CASH_OUT génèrent des fraudes |
| **Dice** | 44 fraudes concentrées la nuit, 63 le matin (mais taux nuit 8× plus élevé) |
| **Drill-down** | 107 fraudes au total → 53 TRANSFER + 54 CASH_OUT |
| **Roll-up catégorie** | 100% des fraudes dans la catégorie 'interne' |
| **Mesure calculée** | Score de risque composite permet de prioriser les alertes |
| **Top-N** | Destinataire C803116137 : 11 montants aberrants reçus sur 43 transactions |
| **Évolution** | Pic de fraude au Step 4 (nuit) : 1,77% |
